In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd



from scipy.stats import permutation_test
sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")

# Ahora importa la función
from print5 import print5

In [2]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_pat

In [3]:


channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]

channels_mag=channels_mag.tolist()
print5(channels_mag)
indice_channels_efectivos = channels[channels[f"canal_efectivo_{modality}"].notna()]["indice"]
indice_channels_efectivos=indice_channels_efectivos.tolist()
# del channels



#lectura de acw
acw_0_df = pd.read_pickle(ACW_path /f"acw_0_df.pickle")

##creacion dataframe acw_0
# acw_all = pd.read_pickle(ACW_path /f"autocorrelation_subjects_all.pickle")
# acw_0_df = acw_all[["Subject", "Condition", "Epoch",
#                    "Elect", "acw_0_elect_all_epoch_all"]]
# acw_0_df.to_pickle(ACW_path / f"acw_0_df.pickle")

# acw_0_df = acw_all[["Subject", "Condition", "Epoch",
#                    "Elect", "acw_0_elect_all_epoch_all"]]
# acw_0_df.to_pickle(ACW_path / f"acw_0_df.pickle")

# del acw_all, del acw_0_df
##valores de las columnas
condition = acw_0_df["Condition"].unique()
print("Condiciones en los datos:", condition)

subjects = acw_0_df["Subject"].unique()
print("Sujetos en los datos:", subjects)

elect_all= acw_0_df["Elect"].unique()
print("sensores en los datos:", elect_all)

epochs_all= acw_0_df["Epoch"].unique()
print("Epochs en los datos:", epochs_all)


['MLC11-4304', 'MLC12-4304', 'MLC13-4304', 'MLC14-4304', 'MLC15-4304', 'MLC16-4304', 'MLC17-4304', 'MLC21-4304', 'MLC22-4304', 'MLC23-4304', 'MLC24-4304', 'MLC25-4304', 'MLC31-4304', 'MLC32-4304', 'MLC41-4304', 'MLC42-4304', 'MLC51-4304', 'MLC52-4304', 'MLC53-4304', 'MLC54-4304', 'MLC55-4304', 'MLC61-4304', 'MLC62-4304', 'MLC63-4304', 'MLF11-4304', 'MLF12-4304', 'MLF13-4304', 'MLF14-4304', 'MLF21-4304', 'MLF22-4304', 'MLF23-4304', 'MLF24-4304', 'MLF25-4304', 'MLF31-4304', 'MLF32-4304', 'MLF33-4304', 'MLF34-4304', 'MLF35-4304', 'MLF41-4304', 'MLF42-4304', 'MLF43-4304', 'MLF44-4304', 'MLF45-4304', 'MLF46-4304', 'MLF51-4304', 'MLF52-4304', 'MLF53-4304', 'MLF54-4304', 'MLF55-4304', 'MLF56-4304', 'MLF61-4304', 'MLF63-4304', 'MLF64-4304', 'MLF65-4304', 'MLF66-4304', 'MLF67-4304', 'MLO11-4304', 'MLO12-4304', 'MLO13-4304', 'MLO14-4304', 'MLO21-4304', 'MLO22-4304', 'MLO23-4304', 'MLO24-4304', 'MLO31-4304', 'MLO32-4304', 'MLO33-4304', 'MLO34-4304', 'MLO41-4304', 'MLO42-4304', 'MLO43-4304', 'MLO4

In [4]:
#lectura de los diccionarios ISC_zinnen y ISC_woorden
dict_woorden_block = pd.read_pickle(ISC_block_path /f"dict_woorden_block.pkl")
dict_zinnen_block = pd.read_pickle(ISC_block_path /f"dict_zinnen_block.pkl")

In [5]:
##channels significant adjusted in each condition
numbers_channels_woorden= dict_woorden_block["significant_channels_adjusted_woorden"]
numbers_channels_zinnen= dict_zinnen_block["significant_channels_adjusted_zinnen"]
print("numbers_channels_woorden", numbers_channels_woorden, "len",len(numbers_channels_woorden))

print("numbers_channels_zinnen", numbers_channels_zinnen, "len", len(numbers_channels_zinnen))


numbers_channels_woorden [  1  14  23  46  56  59  60  63  67  71  82  83  84  86  87  88  89  90
  92  93  94 112 118 122 133 140 144 145 150 180 184 192 210 213 215 221
 224 237 240 254 256 257 269 270] len 44
numbers_channels_zinnen [ 10  13  29  31  32  36  37  40  41  45  46  56  60  72  79  82  83  84
  86  87  88  89  94 105 106 112 117 118 123 135 136 140 141 143 144 145
 155 167 173 199 200 211 212 215 216 219 223 228 229 230 234 235 242 243
 244 250 251 256] len 58


In [6]:
##channels significant adjusted in each condition
numbers_channels_woorden= dict_woorden_block["significant_channels_adjusted_woorden"]
numbers_channels_zinnen= dict_zinnen_block["significant_channels_adjusted_zinnen"]
print("numbers_channels_woorden", numbers_channels_woorden, "numbers_channels_zinnen", numbers_channels_zinnen)
print("len(numbers_channels_woorden)",len(numbers_channels_woorden),  "len(numbers_channels_zinnen)",len(numbers_channels_zinnen))

names_channels_woorden=[channels_mag[i] for i in numbers_channels_woorden]
names_channels_zinnen=[channels_mag[i] for i in numbers_channels_zinnen]

print("names_channels_woorden", names_channels_woorden)
print("names_channels_zinnen", names_channels_zinnen)

#establecimiento de canales palabras, canales frases y canales mixtos
numbers_channels_only_woorden = list(
    set(numbers_channels_woorden) - set(numbers_channels_zinnen)
)

numbers_channels_only_zinnen = list(
    set(numbers_channels_zinnen) - set(numbers_channels_woorden)
)

number_channels_intersection = list(
    set(numbers_channels_woorden) & set(numbers_channels_zinnen)
)

print(f"len(significant_channels_only_woorden): {len(numbers_channels_only_woorden)},\
      len(significant_channels_only_zinnen): {len(numbers_channels_only_zinnen)},\
      len(significant_channels_intersection) {len(number_channels_intersection)}")

#create list of names of channels

names_channels_only_woorden = [channels_mag[i] for i in numbers_channels_only_woorden]
names_channels_only_zinnen = [channels_mag[i] for i in numbers_channels_only_zinnen]
names_channels_intersection = [channels_mag[i] for i in number_channels_intersection]

numbers_channels_woorden [  1  14  23  46  56  59  60  63  67  71  82  83  84  86  87  88  89  90
  92  93  94 112 118 122 133 140 144 145 150 180 184 192 210 213 215 221
 224 237 240 254 256 257 269 270] numbers_channels_zinnen [ 10  13  29  31  32  36  37  40  41  45  46  56  60  72  79  82  83  84
  86  87  88  89  94 105 106 112 117 118 123 135 136 140 141 143 144 145
 155 167 173 199 200 211 212 215 216 219 223 228 229 230 234 235 242 243
 244 250 251 256]
len(numbers_channels_woorden) 44 len(numbers_channels_zinnen) 58
names_channels_woorden ['MLC12-4304', 'MLC41-4304', 'MLC63-4304', 'MLF53-4304', 'MLO11-4304', 'MLO14-4304', 'MLO21-4304', 'MLO24-4304', 'MLO34-4304', 'MLO44-4304', 'MLP33-4304', 'MLP34-4304', 'MLP35-4304', 'MLP42-4304', 'MLP43-4304', 'MLP44-4304', 'MLP45-4304', 'MLP51-4304', 'MLP53-4304', 'MLP54-4304', 'MLP55-4304', 'MLT33-4304', 'MLT43-4304', 'MLT47-4304', 'MRC14-4304', 'MRC24-4304', 'MRC41-4304', 'MRC42-4304', 'MRC55-4304', 'MRF61-4304', 'MRF65-4304', 'MRO22-4304

## Posibles comparaciones estadísticas
len(significant_channels_only_woorden): 25, len(significant_channels_only_zinnen): 92, len(significant_channels_intersection) 88

Primer problema, el numero de canales zinnen es claramente superior al de woorden, big problem, aunque la interseccion es alta, tal como esperaríamos. Seguramente esto sea porque hay ruido 

- comparación ACW-50 en zinnen entre estos canales zinnen y canales woorden  + comapracion en woorden de lo mismo

- comparación entre solo zinnen con intersection  en woorden y en zinnen (aunque ojo, esto es básicamente aceptar que los canales solo Zinnen son ruido)

- comapración en el promedio general de acw en ambas condiciones de los canalaes zinnen vs promedio general de canales woorden

- comparación de la intersección vs canales no signficativos: no sé muy bien como interpretar esto

- 

In [ ]:
channels_type=["names_channels_only_woorden", "names_channels_only_zinnen", "names_channels_intersection"]



##matrices de valores de acw_0, divididas por condicion y tipo de channel
X_zinnen_ch_zinnen = []
X_zinnen_ch_woorden = []
X_zinnen_ch_intersection = []

X_woorden_ch_zinnen = []
X_woorden_ch_woorden = []
X_woorden_ch_intersection = []

##Filtras la condicion
for cond in condition:
    acw_0_condition_df=acw_0_df[acw_0_df["Condition"] == f"{cond}"]

    #filtras por tipo de canal only zinnen o only woorden
    for ch_type in channels_type:

        #bucle if para seleccionar el tipo de canal
        if ch_type == "names_channels_only_zinnen":
            elects=names_channels_only_zinnen
            print(ch_type, elects)
        if ch_type == "names_channels_only_woorden":
            elects=names_channels_only_woorden
            print(ch_type, elects)
        if ch_type == "names_channels_intersection":
            elects=names_channels_intersection
            print(ch_type, elects)

        

NameError: name 'names_channels_only_woorden' is not defined

In [15]:
acw_0_condition_df

,Subject,Condition,Epoch,Elect,acw_0_elect_all_epoch_all
6552,sub-V1001,woorden,0,MLC11-4304,0.346667
6553,sub-V1001,woorden,0,MLC12-4304,0.350000
6554,sub-V1001,woorden,0,MLC13-4304,0.366667
6555,sub-V1001,woorden,0,MLC14-4304,0.330000
6556,sub-V1001,woorden,0,MLC15-4304,0.246667
...,...,...,...,...,...
187000,sub-V1024,woorden,23,MZF03-4304,0.050000
187001,sub-V1024,woorden,23,MZO01-4304,0.036667
187002,sub-V1024,woorden,23,MZO02-4304,0.040000
187003,sub-V1024,woorden,23,MZO03-4304,0.046667


In [14]:
acw_0_condition_ch_type_df


,Subject,Condition,Epoch,Elect,acw_0_elect_all_epoch_all
6598,sub-V1001,woorden,0,MLF53-4304,0.150000
6608,sub-V1001,woorden,0,MLO11-4304,0.150000
6612,sub-V1001,woorden,0,MLO21-4304,0.150000
6634,sub-V1001,woorden,0,MLP33-4304,0.250000
6635,sub-V1001,woorden,0,MLP34-4304,0.250000
...,...,...,...,...,...
186872,sub-V1024,woorden,23,MRC24-4304,0.046667
186876,sub-V1024,woorden,23,MRC41-4304,0.046667
186877,sub-V1024,woorden,23,MRC42-4304,0.040000
186947,sub-V1024,woorden,23,MRP35-4304,0.043333


In [7]:
##selección de valores


channels_type=["names_channels_only_woorden", "names_channels_only_zinnen", "names_channels_intersection"]

##matrices de valores de acw_0, divididas por condicion y tipo de channel
X_zinnen_ch_zinnen = []
X_zinnen_ch_woorden = []
X_zinnen_ch_intersection = []

X_woorden_ch_zinnen = []
X_woorden_ch_woorden = []
X_woorden_ch_intersection = []

##Filtras la condicion
for cond in condition:
    acw_0_condition_df=acw_0_df[acw_0_df["Condition"] == f"{cond}"]

    #filtras por tipo de canal only zinnen o only woorden
    for ch_type in channels_type:

        #bucle if para seleccionar el tipo de canal
        if ch_type == "names_channels_only_zinnen":
            elects=names_channels_only_zinnen
        if ch_type == "names_channels_only_woorden":
            elects=names_channels_only_woorden
        if ch_type == "names_channels_intersection":
            elects=names_channels_intersection

        acw_0_condition_ch_type_df=acw_0_condition_df[acw_0_condition_df["Elect"].isin(elects)]
        #filtras por sujeto 
        for subj in subjects:
                #creacion de lista de valores de acw_0 para cada sujeto
                acw_0_epoch_list= []

                #filtras por sujeto
                for epoch in epochs_all:
                    # Extraer el valor de acw_0 para la combinación actual
                    try:
                    #coges el valor de acw_0 para el sujeto y el epoch
                        print(f"subj:{subj},epoch, {epoch}")
                        acw_0_elect_all_epoch_all = acw_0_condition_ch_type_df[(acw_0_condition_ch_type_df["Subject"] == subj) & (acw_0_condition_ch_type_df["Epoch"] == epoch)]["acw_0_elect_all_epoch_all"]
                        if not acw_0_elect_all_epoch_all.empty:
                            acw_0_epoch_list.append(acw_0_elect_all_epoch_all)

                    except Exception as e:
                        print(e, "probablemente faltaban epochs en algunos sujetos")
                        continue
                # Convertir la lista a un array de numpy
                acw_0_epoch_array = np.array(acw_0_epoch_list)
                #take the mean on epochs
                acw_0_epoch_mean = np.mean(acw_0_epoch_array, axis=0)

                #add it to the different lists
                if cond == "zinnen":
                    if ch_type == "names_channels_only_zinnen":
                        X_zinnen_ch_zinnen.append(acw_0_epoch_mean)
                    elif ch_type == "names_channels_only_woorden":
                        X_zinnen_ch_woorden.append(acw_0_epoch_mean)
                    elif ch_type == "names_channels_intersection":
                        X_zinnen_ch_intersection.append(acw_0_epoch_mean)
                if cond == "woorden":
                    if ch_type == "names_channels_only_zinnen":
                        X_woorden_ch_zinnen.append(acw_0_epoch_mean)
                    elif ch_type == "names_channels_only_woorden":
                        X_woorden_ch_woorden.append(acw_0_epoch_mean)
                    elif ch_type == "names_channels_intersection":
                        X_woorden_ch_intersection.append(acw_0_epoch_mean)


X_zinnen_ch_zinnen = np.array(X_zinnen_ch_zinnen)
X_zinnen_ch_woorden = np.array(X_zinnen_ch_woorden)
X_zinnen_ch_intersection = np.array(X_zinnen_ch_intersection)

X_woorden_ch_zinnen = np.array(X_woorden_ch_zinnen)
X_woorden_ch_woorden = np.array(X_woorden_ch_woorden)
X_woorden_ch_intersection = np.array(X_woorden_ch_intersection)

print("Shapes de las matrices de valores acw_0:\n")

print("▶ Condición: ZINNEN")
print("  - Canales only_zinnen      :", X_zinnen_ch_zinnen.shape)
print("  - Canales only_woorden     :", X_zinnen_ch_woorden.shape)
print("  - Canales intersection     :", X_zinnen_ch_intersection.shape)

print("\n▶ Condición: WOORDEN")
print("  - Canales only_zinnen      :", X_woorden_ch_zinnen.shape)
print("  - Canales only_woorden     :", X_woorden_ch_woorden.shape)
print("  - Canales intersection     :", X_woorden_ch_intersection.shape)


subj:sub-V1001,epoch, 0
subj:sub-V1001,epoch, 1
subj:sub-V1001,epoch, 2
subj:sub-V1001,epoch, 3
subj:sub-V1001,epoch, 4
subj:sub-V1001,epoch, 5
subj:sub-V1001,epoch, 6
subj:sub-V1001,epoch, 7
subj:sub-V1001,epoch, 8
subj:sub-V1001,epoch, 9
subj:sub-V1001,epoch, 10
subj:sub-V1001,epoch, 11
subj:sub-V1001,epoch, 12
subj:sub-V1001,epoch, 13
subj:sub-V1001,epoch, 14
subj:sub-V1001,epoch, 15
subj:sub-V1001,epoch, 16
subj:sub-V1001,epoch, 17
subj:sub-V1001,epoch, 18
subj:sub-V1001,epoch, 19
subj:sub-V1001,epoch, 20
subj:sub-V1001,epoch, 21
subj:sub-V1001,epoch, 22
subj:sub-V1001,epoch, 23
subj:sub-V1002,epoch, 0
subj:sub-V1002,epoch, 1
subj:sub-V1002,epoch, 2
subj:sub-V1002,epoch, 3
subj:sub-V1002,epoch, 4
subj:sub-V1002,epoch, 5
subj:sub-V1002,epoch, 6
subj:sub-V1002,epoch, 7
subj:sub-V1002,epoch, 8
subj:sub-V1002,epoch, 9
subj:sub-V1002,epoch, 10
subj:sub-V1002,epoch, 11
subj:sub-V1002,epoch, 12
subj:sub-V1002,epoch, 13
subj:sub-V1002,epoch, 14
subj:sub-V1002,epoch, 15
subj:sub-V1002,epoch

In [8]:

## permutaciones para comparar los valores de acw_0 entre los canales significativos de cada condicion
X_zinnen_ch_zinnen_mean = X_zinnen_ch_zinnen.mean(axis=1)
X_zinnen_ch_woorden_mean = X_zinnen_ch_woorden.mean(axis=1)
print(f"X_zinnen_ch_zinnen_mean, {X_zinnen_ch_zinnen_mean.shape}")	
print(f"X_zinnen_ch_woorden_mean, {X_zinnen_ch_woorden_mean.shape}")	



X_woorden_ch_zinnen_mean = X_woorden_ch_zinnen.mean(axis=1)
X_woorden_ch_woorden_mean = X_woorden_ch_woorden.mean(axis=1)
print(f"X_woorden_ch_zinnen_mean, {X_woorden_ch_zinnen_mean.shape}")	
print(f"X_woorden_ch_woorden_mean, {X_woorden_ch_woorden_mean.shape}")


X_zinnen_ch_intersection_mean = X_zinnen_ch_intersection.mean(axis=1)
X_woorden_ch_intersection_mean = X_woorden_ch_intersection.mean(axis=1)

print(f"X_zinnen_ch_intersection_mean, {X_zinnen_ch_intersection_mean.shape}")
print(f"X_woorden_ch_intersection_mean, {X_woorden_ch_intersection_mean.shape}")


# Crear el diccionario con nombres descriptivos
data_dict = {
    "X_zinnen_ch_zinnen_mean": X_zinnen_ch_zinnen_mean,
    "X_zinnen_ch_woorden_mean": X_zinnen_ch_woorden_mean,
    "X_woorden_ch_zinnen_mean": X_woorden_ch_zinnen_mean,
    "X_woorden_ch_woorden_mean": X_woorden_ch_woorden_mean,
    "X_zinnen_ch_intersection_mean": X_zinnen_ch_intersection_mean,
    "X_woorden_ch_intersection_mean": X_woorden_ch_intersection_mean,
}

X_zinnen_ch_zinnen_mean, (15,)
X_zinnen_ch_woorden_mean, (15,)
X_woorden_ch_zinnen_mean, (15,)
X_woorden_ch_woorden_mean, (15,)
X_zinnen_ch_intersection_mean, (15,)
X_woorden_ch_intersection_mean, (15,)


# Comparaciones Zinnen only vs intersection


In [20]:

from scipy.stats import permutation_test

##dependent condition
def paired_statistic(diff, _):
    return np.mean(diff)


# Independent condition
def diff_means(x, y):
    return np.mean(x) - np.mean(y)

# if we assume dependency

In [19]:


experimental_condition=["zinnen", "woorden"]
type_channel=["intersection_mean", "woorden_mean"]
print("DEPENDENT analysis")

for exp in experimental_condition:
    print(f"in {exp} condition")
    for ch in type_channel:
        print(f"for differences between zinnen only and {ch}:")
        # print(f"Processing {exp} for differences between zinenn and {ch}...")
        ##notice that in this comparison X is alwas ch_zinnnen_mean
        x= data_dict[f"X_{exp}_ch_zinnen_mean"]
        ##notice that in this comparison y is  ch_woorden_mean or ch_intersection_mean
        y=data_dict[f"X_{exp}_ch_{ch}"]

        diff=x-y
        # Permutation test
        res = permutation_test(
            (diff, np.zeros_like(diff)),
            #
            statistic=paired_statistic,
            vectorized=False,
            n_resamples=10000,
            alternative='greater', 
            random_state=42
        )

        # Cohen's d
        mean_diff = np.mean(diff)
        std_diff = np.std(diff, ddof=1)
        cohens_d = mean_diff / std_diff

        print(f"   Statistic value: {res.statistic:.6f}")
        print(f"   p-value        : {res.pvalue:.5f}")
        print(f"   Cohen's d      : {cohens_d:.3f}")

DEPENDENT analysis
in zinnen condition
for differences between zinnen only and intersection_mean:
   Statistic value: 0.044392
   p-value        : 0.00010
   Cohen's d      : 1.047
for differences between zinnen only and woorden_mean:
   Statistic value: 0.067392
   p-value        : 0.00010
   Cohen's d      : 3.019
in woorden condition
for differences between zinnen only and intersection_mean:
   Statistic value: 0.038293
   p-value        : 0.00010
   Cohen's d      : 0.998
for differences between zinnen only and woorden_mean:
   Statistic value: 0.064008
   p-value        : 0.00010
   Cohen's d      : 3.369


In [ ]:
##if we assume independency


##comapraciones en la condicion zinnen
experimental_condition=["zinnen", "woorden"]
type_channel=["intersection_mean", "woorden_mean"]

print("INDEPENDENT analysis")
for exp in experimental_condition:
    print(f"in {exp} condition")
    for ch in type_channel:
        print(f"for differences between zinenn and {ch}:")
        # print(f"Processing {exp} for differences between zinenn and {ch}...")
        ##notice that in this comparison X is alwas ch_zinnnen_mean
        x= data_dict[f"X_{exp}_ch_zinnen_mean"]
        ##notice that in this comparison y is  ch_woorden_mean or ch_intersection_mean
        y=data_dict[f"X_{exp}_ch_{ch}"]

        # Hacemos la prueba de permutación
        res = permutation_test(
            (x, y),
            statistic=diff_means,
            vectorized=False,
            n_resamples=10000,
            alternative='greater', 
            random_state=42
        )

        # Calcular Cohen's d (independent)
        mean_x, mean_y = np.mean(x), np.mean(y)
        std_x, std_y = np.std(x, ddof=1), np.std(y, ddof=1)
        n_x, n_y = len(x), len(y)

        pooled_std = np.sqrt(((n_x - 1) * std_x**2 + (n_y - 1) * std_y**2) / (n_x + n_y - 2))
        cohens_d = (mean_x - mean_y) / pooled_std

        # Imprimir resultados
        print(f"   Statistic value: {res.statistic:.6f}")
        print(f"   p-value        : {res.pvalue:.5f}")
        print(f"   Cohen's d      : {cohens_d:.3f}")


## using average in both conditions

In [21]:

experimental_condition=["zinnen", "woorden"]
type_channel=["intersection_mean", "woorden_mean"]
print("DEPENDENT analysis")


for ch in type_channel:
    print(f"for differences between zinnen only and {ch}:")
    # print(f"Processing {exp} for differences between zinenn and {ch}...")
    ##notice that in this comparison X is always ch_zinnnen_mean
    x= (data_dict[f"X_zinnen_ch_zinnen_mean"] + data_dict[f"X_woorden_ch_zinnen_mean"])/2

    ##notice that in this comparison y is  ch_woorden_mean or ch_intersection_mean
    y=(data_dict[f"X_zinnen_ch_{ch}"] + data_dict[f"X_woorden_ch_{ch}"])/2

    diff=x-y
    # Permutation test
    res = permutation_test(
        (diff, np.zeros_like(diff)),
        #
        statistic=paired_statistic,
        vectorized=False,
        n_resamples=10000,
        alternative='greater', 
        random_state=42
    )
    # Cohen's d
    mean_diff = np.mean(diff)
    std_diff = np.std(diff, ddof=1)
    cohens_d = mean_diff / std_diff

    print(f"   Statistic value: {res.statistic:.6f}")
    print(f"   p-value        : {res.pvalue:.5f}")
    print(f"   Cohen's d      : {cohens_d:.3f}")

DEPENDENT analysis
for differences between zinnen only and intersection_mean:
   Statistic value: 0.041342
   p-value        : 0.00010
   Cohen's d      : 1.026
for differences between zinnen only and woorden_mean:
   Statistic value: 0.065700
   p-value        : 0.00010
   Cohen's d      : 3.255


In [22]:

experimental_condition=["zinnen", "woorden"]
type_channel=["intersection_mean", "woorden_mean"]
print("INdependent analysis")


for ch in type_channel:
    print(f"for differences between zinnen only and {ch}:")
    # print(f"Processing {exp} for differences between zinenn and {ch}...")
    ##notice that in this comparison X is always ch_zinnnen_mean
    x= (data_dict[f"X_zinnen_ch_zinnen_mean"] + data_dict[f"X_woorden_ch_zinnen_mean"])/2

    ##notice that in this comparison y is  ch_woorden_mean or ch_intersection_mean
    y=(data_dict[f"X_zinnen_ch_{ch}"] + data_dict[f"X_woorden_ch_{ch}"])/2

    diff=x-y
    # Permutation test
    res = permutation_test(
        (diff, np.zeros_like(diff)),
        #
        statistic=diff_means,
        vectorized=False,
        n_resamples=10000,
        alternative='greater', 
        random_state=42
    )

    # Calcular Cohen's d (independent)
    mean_x, mean_y = np.mean(x), np.mean(y)
    std_x, std_y = np.std(x, ddof=1), np.std(y, ddof=1)
    n_x, n_y = len(x), len(y)

    pooled_std = np.sqrt(((n_x - 1) * std_x**2 + (n_y - 1) * std_y**2) / (n_x + n_y - 2))
    cohens_d = (mean_x - mean_y) / pooled_std

    # Imprimir resultados
    print(f"   Statistic value: {res.statistic:.6f}")
    print(f"   p-value        : {res.pvalue:.5f}")
    print(f"   Cohen's d      : {cohens_d:.3f}")

INdependent analysis
for differences between zinnen only and intersection_mean:
   Statistic value: 0.041342
   p-value        : 0.00010
   Cohen's d      : 0.516
for differences between zinnen only and woorden_mean:
   Statistic value: 0.065700
   p-value        : 0.00010
   Cohen's d      : 0.970


# using flatten

please note thet this is wrong

In [ ]:
# ## permutaciones para comparar los valores de acw_0 entre los canales significativos de cada condicion
# X_zinnen_ch_zinnen_flatten = X_zinnen_ch_zinnen.flatten(axis=1)
# X_zinnen_ch_woorden_flatten = X_zinnen_ch_woorden.flatten(axis=1)
# print(f"X_zinnen_ch_zinnen_flatten, {X_zinnen_ch_zinnen_flatten.shape}")	
# print(f"X_zinnen_ch_woorden_flatten, {X_zinnen_ch_woorden_flatten.shape}")	



# X_woorden_ch_zinnen_flatten = X_woorden_ch_zinnen.flatten(axis=1)
# X_woorden_ch_woorden_flatten = X_woorden_ch_woorden.flatten(axis=1)
# print(f"X_woorden_ch_zinnen_flatten, {X_woorden_ch_zinnen_flatten.shape}")	
# print(f"X_woorden_ch_woorden_flatten, {X_woorden_ch_woorden_flatten.shape}")


# X_zinnen_ch_intersection_flatten = X_zinnen_ch_intersection.flatten(axis=1)
# X_woorden_ch_intersection_flatten = X_woorden_ch_intersection.flatten(axis=1)

# print(f"X_zinnen_ch_intersection_flatten, {X_zinnen_ch_intersection_flatten.shape}")
# print(f"X_woorden_ch_intersection_flatten, {X_woorden_ch_intersection_flatten.shape}")


# # Crear el diccionario con nombres descriptivos
# data_dict = {
#     "X_zinnen_ch_zinnen_flatten": X_zinnen_ch_zinnen_flatten,
#     "X_zinnen_ch_woorden_flatten": X_zinnen_ch_woorden_flatten,
#     "X_woorden_ch_zinnen_flatten": X_woorden_ch_zinnen_flatten,
#     "X_woorden_ch_woorden_flatten": X_woorden_ch_woorden_flatten,
#     "X_zinnen_ch_intersection_flatten": X_zinnen_ch_intersection_flatten,
#     "X_woorden_ch_intersection_flatten": X_woorden_ch_intersection_flatten}

# comparison of word sentence channels in both conditions

Before we compared differences of word and sentence channels. Now we are going to compare the channels with themselves in different experimental conditions

In [23]:
## statistical test

## REVIEWW THISSS
experimental_condition=["zinnen", "woorden"]
type_channel=["zinnen_mean","intersection_mean", "woorden_mean"]
print("DEPENDENT analysis")


for ch in type_channel:
    print(f"for differences in ch {ch}  between zinnen and word condition:")
    # print(f"Processing {exp} for differences between zinenn and {ch}...")

    #ch value in zinnen condition
    x= data_dict[f"X_zinnen_ch_{ch}"]

    y=data_dict[f"X_woorden_ch_{ch}"]

    print(f"X is: X_zinnen_ch_{ch}", x.shape)
    print(f"Y is: X_woorden_ch_{ch}", y.shape)


    diff=x-y
    # Permutation test
    res = permutation_test(
        (diff, np.zeros_like(diff)),
        statistic=paired_statistic,
        vectorized=False,
        n_resamples=10000,
        alternative='greater', 
        random_state=42
    )

    # Calcular Cohen's d para muestras emparejadas
    mean_diff = np.mean(diff)
    std_diff = np.std(diff, ddof=1)
    cohens_d = mean_diff / std_diff

    # Imprimir resultados
    print(f"   Statistic value: {res.statistic:.6f}")
    print(f"   p-value        : {res.pvalue:.5f}")
    print(f"   Cohen's d      : {cohens_d:.3f}")

DEPENDENT analysis
for differences in ch zinnen_mean  between zinnen and word condition:
X is: X_zinnen_ch_zinnen_mean (15,)
Y is: X_woorden_ch_zinnen_mean (15,)
   Statistic value: 0.006116
   p-value        : 0.02250
   Cohen's d      : 0.533
for differences in ch intersection_mean  between zinnen and word condition:
X is: X_zinnen_ch_intersection_mean (15,)
Y is: X_woorden_ch_intersection_mean (15,)
   Statistic value: 0.000016
   p-value        : 0.49715
   Cohen's d      : 0.002
for differences in ch woorden_mean  between zinnen and word condition:
X is: X_zinnen_ch_woorden_mean (15,)
Y is: X_woorden_ch_woorden_mean (15,)
   Statistic value: 0.002732
   p-value        : 0.15028
   Cohen's d      : 0.275
